In [69]:
import importlib
import RAG_model.ingestion.config as config

print(config.__file__)  # confirms which config.py Python loaded

importlib.reload(config)

print(config.GENERATION_MODEL)

C:\Users\david\Documents\Projects\crypto_chatbot_rag\src\RAG_model\ingestion\config.py
openai/gpt-4o-mini


In [70]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(r"C:\Users\david\Documents\Projects\crypto_chatbot_rag")
SRC_PATH = PROJECT_ROOT / "src"

assert (SRC_PATH / "RAG_model").is_dir(), f"Could not find package at: {SRC_PATH}"

sys.path.insert(0, str(SRC_PATH))

print("Added to Python path:", SRC_PATH)

Added to Python path: C:\Users\david\Documents\Projects\crypto_chatbot_rag\src


In [71]:
from RAG_model.ingestion.config import EMBEDDINGS_MODEL, COLLECTION_NAME, DB_PATH_NAME, GENERATION_MODEL
from RAG_model.ingestion.embedding import create_openrouter_client
from qdrant_client import QdrantClient
from tenacity import retry, wait_exponential, stop_after_attempt
from time import perf_counter

In [72]:
wait = wait_exponential(multiplier=1, min=10, max=240)


In [73]:
qdrant_client = QdrantClient(path = str(DB_PATH_NAME))


In [74]:
RETRIEVAL_K = 5

In [75]:
client = create_openrouter_client()

In [76]:
def fetch_context(question, retrieval_k = RETRIEVAL_K):
    """Embed a question and retrieve its top-k most similar chunks."""
    
    # Embedd the question with the same embedding model
    query = client.embeddings.create(model = EMBEDDINGS_MODEL, input=[question]).data[0].embedding
    
    # Return results from the database based on the question
    results = qdrant_client.query_points(
        collection_name= COLLECTION_NAME,
        query = query,
        limit= retrieval_k,
        with_payload= True)
        
    # Convert the payload to the right format 
    
    chunks = []
    
    for point in results.points:
        payload = point.payload or {}

        chunks.append(
            {
                "chunk_id": payload["chunk_id"],
                "chunk_text": payload["chunk_text"],
                "ticker": payload["ticker"],
                "filing_date": payload["filing_date"],
                "section_title": payload["chunk_title"],
                "source_url": payload["source_url"],
                "score": point.score,
            }
        )

    return chunks

In [77]:
results = fetch_context("What is FETH?")
for rank, result in enumerate(results, start=1):
    print(f"\n--- Result {rank} ---")
    print(f"Score: {result['score']:.4f}")
    print(f"Chunk ID: {result['chunk_id']}")
    print(f"Ticker: {result['ticker']}")
    print(f"Section: {result['section_title']}")
    print("Result:")
    print(result["chunk_text"][:600])


--- Result 1 ---
Score: 0.5441
Chunk ID: 0000950170-25-039374*section-5*table-0015
Ticker: FETH
Section: Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities
Result:
Section: Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities
Table ID: table-0015

Trust | Commencement of Operations | Ticker Symbol | Name of each exchange on which registered
Fidelity Ethereum Fund | July 23, 2024 | FETH | Cboe BZX Exchange, Inc.

--- Result 2 ---
Score: 0.5410
Chunk ID: 0001193125-26-071486*section-5*table-0022
Ticker: FETH
Section: Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities
Result:
Section: Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities
Table ID: table-0022

Trust | Commencement of Operations | Ticker Symbol | Name of each exchange on which registered
Fidelity Eth

In [78]:
def make_rag_messages(question, history, chunks):
    """Build chat messages for the RAG answer step: system (with context) + history + user question."""
    context = "\n\n".join(
        f"-Extract from:\n \
           [SOURCE:{chunk['chunk_id']:}]\n \
            - TICKER: {chunk['ticker']}\n \
            - Section:{chunk['section_title']}\n \
            - Filing Date:{chunk['filing_date']}\n \
            - SEC URL{chunk['source_url']}] \n \
            - Content: \n {chunk['chunk_text']}  " for chunk in chunks)
    
    system_prompt = f"""
    You are a knowledgeable, friendly assistant that helps with question regard some specific companies extracting only the information available from the SEC 10-K and 10-Q forms those companies submmit.
    You are chatting with a user about Answer questions about these SEC filing entities: IBIT, ETHA, FBTC, FETH, GBTC, ETHE. If the question is outside this corpus, explain that the corpus does
    not contain evidence to answer it.
    Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
    Answer using only the supplied context.

    If the context does not contain enough information, say:
    "I could not find enough evidence in the retrieved SEC filings."

    Cite the supplied source IDs for every factual claim using:
    [SOURCE: chunk_id]

    Never invent a source ID or cite a source that was not supplied.
    
    Do not infer, calculate, or compare values unless the retrieved context
    contains all evidence needed. Otherwise say that there is not enough evidence.
    
    For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
    {context}

    With this context, please answer the user's question. Be accurate, relevant and complete.
    """
    
    return (
        [{"role": "system",
          "content": system_prompt}]
        + history
        + [{"role": "user",
            "content": question}]
        )

In [79]:
@retry(wait=wait,
       stop=stop_after_attempt(4))
def answer(question: str, history:list[dict] | None = None) -> tuple[str, list[dict], dict[str, float]]:
    
    if history is None:
        history = []
    
    total_start = perf_counter()
    
    
    retrieval_start = perf_counter()
    chunks = fetch_context(question)
    retrieval_end = perf_counter()
    
    prompt_start = perf_counter()
    messages = make_rag_messages(question, history, chunks)
    prompt_end = perf_counter()
    
    generation_start = perf_counter()
    response = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=messages,
    )
    generation_end = perf_counter()
    
    answer_text = response.choices[0].message.content
    
    latency_ms = {
        "retrieval": (retrieval_end - retrieval_start) * 1000,
        "prompt_building": (prompt_end - prompt_start) * 1000,
        "generation": (generation_end - generation_start) * 1000,
        "total": (generation_end - total_start) * 1000,
    }
    
    return answer_text, chunks, latency_ms
    

In [83]:
"""
Format the answer to be in the right format with all the necessary fields

Print the following during development:

- User question.
- Retrieved chunk texts.
- Similarity scores.
- Final context.
- Answer.
- Citations.
- Latency.
"""

'\nFormat the answer to be in the right format with all the necessary fields\n\nPrint the following during development:\n\n- User question.\n- Retrieved chunk texts.\n- Similarity scores.\n- Final context.\n- Answer.\n- Citations.\n- Latency.\n'

In [80]:
answer_text, chunks, metrics = answer("What is the ticker symbol for Fidelity Ethereum Fund?")
print(answer_text)

The ticker symbol for Fidelity Ethereum Fund is FETH [SOURCE:0000950170-25-039374*section-5*table-0015].


In [81]:
print(metrics)

{'retrieval': 283.0915999948047, 'prompt_building': 0.04169999738223851, 'generation': 1810.5744000058621, 'total': 2093.708100001095}


In [82]:
client.close()
qdrant_client.close()
print("Qdrant client closed.")

Qdrant client closed.
